# V122 - Pseudo-relevance feedback

Pre-registered plan: classic IR expands queries with top retrieved documents. Here each user doc expands with the syllabi of its top-3 Ridge modules, then a second Ridge ranks from expanded docs. Expansion and ranking both stay inside training folds, so no label reuse. Same grouped folds under a fresh seed, same locks. If syllabus language bridges chat intent, expansion wins.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 5151
N_SPLITS = 5
GATE_FLOOR = 0.002
TOPK = 3
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v122")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data


## Expand with top-3 syllabi, rerank, gate

Stage one ranks from chat alone per fold. Stage two vectorizes chat plus top syllabi in a joint space and reranks. Folds never mix.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
catalog = pd.read_csv(DATA / "modules_catalog.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in list(train["user_id"])]
syll = [str(catalog.set_index("module_id").loc[m, "description_and_syllabus"]) if m in set(catalog["module_id"]) else "" for m in MODS]

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

VF = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit(docs + syll)
W = VF.transform(docs).toarray()
VF = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit(docs + syll)
W = VF.transform(docs).toarray()
oof_w, fids = np.zeros_like(Y), np.zeros(n_train, int)
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(skf.split(W, strata, fold_groups)):
    for j in range(len(MODS)):
        m = Ridge(alpha=1.0)
        m.fit(W[tr], Y[tr, j])
        oof_w[va, j] = m.predict(W[va])
    fids[va] = fi + 1
exp_all = [docs[u] + " " + " ".join(syll[j] for j in np.argsort(-oof_w[u] if False else oof_w[u])[:TOPK]) for u in range(n_train)]
oof_p = np.zeros_like(Y)
for fi in range(1, N_SPLITS + 1):
    tr, va = np.where(fids != fi)[0], np.where(fids == fi)[0]
    V2 = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit([exp_all[u] for u in tr])
    for j in range(len(MODS)):
        r = Ridge(alpha=1.0)
        r.fit(V2.transform([exp_all[u] for u in tr]).toarray(), Y[tr, j])
        oof_p[va, j] = r.predict(V2.transform([exp_all[u] for u in va]).toarray())
    print(f"fold {fi} done", flush=True)
base = ndcg5(Y, oof_w)
d = ndcg5(Y, oof_p) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fids == k], oof_p[fids == k]).mean() - base[fids == k].mean() for k in range(1, N_SPLITS + 1))
print(f"plain={base.mean():.5f} prf={ndcg5(Y, oof_p).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
print("ADOPT PRF expansion" if (lo > GATE_FLOOR and wf >= 0) else "KEEP plain docs; syllabus expansion adds nothing")

fold 1 done


fold 2 done


fold 3 done


fold 4 done


fold 5 done


plain=0.62479 prf=0.62222 delta=-0.00257 CI95=[-0.00542,+0.00033] worst-fold=-0.00612
KEEP plain docs; syllabus expansion adds nothing
